# Eldercare all-in-one Kaggle demo
Attach the code bundle (including `yolov8n-pose.pt`) and all five Kaggle RGB-5 datasets. This smoke-test runs manifest discovery, short pose adaptation, five limited caches, and temporal training in one session. It does not download raw data.

In [ ]:
%pip install -q --progress-bar off --disable-pip-version-check "ultralytics>=8.3,<9" "lap>=0.5.12" "scikit-learn>=1.4" "PyYAML>=6"

In [ ]:
import importlib.util
from pathlib import Path

candidates = sorted(Path('/kaggle/input').rglob('kaggle_staged.py'))
assert len(candidates) == 1, f'Attach exactly one code Dataset; found: {candidates}'
spec = importlib.util.spec_from_file_location('kaggle_staged', candidates[0])
ks = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ks)
print('Loaded:', candidates[0])

import os
import psutil
import torch
assert torch.cuda.is_available(), (
    'GPU is not available. Open Notebook settings, select Accelerator = GPU, '
    'restart the session, then Run All again.'
)
print('GPU:', torch.cuda.get_device_name(0))

def resources(stage):
    process_gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    system = psutil.virtual_memory()
    gpu_gb = torch.cuda.memory_reserved() / 1024**3
    print(f'[RESOURCE] {stage}: process={process_gb:.2f}GB, '
          f'RAM={system.percent:.1f}%, CUDA_reserved={gpu_gb:.2f}GB')

In [ ]:
resources('before manifest')
SEED = 42
ROOT = Path('/kaggle/working/eldercare_all_in_one')
mounted = ks.validate_kaggle_inputs()
print('Mounted families:', sorted(mounted))
master_manifest = ks.stage00_build_manifest(
    output_dir=ROOT / 'manifest', seed=SEED,
)

In [ ]:
RUN_POSE_FINETUNE = False  # Keep False for the stable all-in-one smoke test
resources('before pose')
if RUN_POSE_FINETUNE:
    pose_checkpoint = ks.stage01_finetune_pose(
        output_dir=ROOT / 'pose', manifest_path=master_manifest,
        mode='auto', frames_per_family=40, pseudo_confidence=0.70,
        pseudo_fps=1.0, gold_repeat=2, epochs=2, image_size=416,
        batch_size=4, workers=0, plots=False, export_openvino=False, seed=SEED,
    )
else:
    pose_checkpoint = ks.stage01_prepare_pose_baseline(output_dir=ROOT / 'pose')
pose_sha_path = pose_checkpoint.parent / 'pose_model.sha256'

In [ ]:
source_limits = {
    'FallVision': 30, 'CAUCAFall': 30, 'URFD': 30,
    'MCFD': 30, 'UCF101': 80,
}
cache_manifests = []
for family, limit in source_limits.items():
    resources(f'before cache {family}')
    print(f'\n=== Cache {family} (max_sources={limit}) ===')
    cache_manifests.append(
        ks.stage_cache_dataset(
            family=family, output_dir=ROOT / 'caches' / family.lower(),
            manifest_path=master_manifest, pose_path=pose_checkpoint,
            pose_sha_path=pose_sha_path, sample_fps=5.0, image_size=320,
            max_sources=limit, minimum_cached_frames=24, seed=SEED,
        )
    )

In [ ]:
resources('before classifier')
classifier_checkpoint = ks.stage07_train_temporal(
    output_dir=ROOT / 'classifier', cache_manifest_paths=cache_manifests,
    pose_model_path=pose_checkpoint,
    sequence_length=24, window_stride=12, max_windows_per_family_split=1_000,
    hidden_dim=48, batch_size=64, epochs=3, learning_rate=2e-3,
    hard_negative_epochs=0,
    workers=0, export_models=False, seed=SEED,
)
print('All-in-one demo complete:', classifier_checkpoint)